In [ ]:
from plotnine import *
import pandas as pd

## Convergence benchmark data

In [ ]:
def process_ebe_data(name):
    df_raw = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_Simwork_ebe_" + name + ".csv")
    ebe_Simwork = (df_raw.rename(columns={"id":"ID"})[["ID","CLint","KbBR","KbMU","KbBO","KbAD","KbRB"]])
    ebe_Simwork["algo"] = "Simwork"
    ebe_Simwork["setting"] = name
    ebe_nlmixr = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_nlmixr_ebe_" + name + ".csv")
    ebe_nlmixr["algo"] = "nlmixr"
    ebe_nlmixr["setting"] = name
    df_raw_2 = pd.read_csv("Benchmark_Data_convergence/Mavoglurant_Simwork_surrogate_ebe_" + name + ".csv")
    ebe_Surrogate = (df_raw_2.rename(columns={"id":"ID"})[["ID","CLint","KbBR","KbMU","KbBO","KbAD","KbRB"]])
    ebe_Surrogate["algo"] = "Simwork with surrogate"
    ebe_Surrogate["setting"] = name
    ebe = pd.concat([ebe_Simwork, ebe_nlmixr, ebe_Surrogate])
    descriptors = ["CLint", "KbBR", "KbMU", "KbBO", "KbAD", "KbRB"]
    ebe = ebe.melt(
        id_vars=["ID", "algo", "setting"], value_vars=descriptors, var_name="descriptor"
    ).reset_index(drop = True)

    return ebe



In [ ]:
ebe_obs_PDU = process_ebe_data("PDU")
ebe_obs_MI = process_ebe_data("MI")
ebe_obs = pd.concat([ebe_obs_PDU, ebe_obs_MI])
display(ebe_obs.head())

In [ ]:
final_fig_size = (12, 10)
color_scale = "Accent"

In [ ]:
p1 = (
    ggplot(ebe_obs, aes(x="value", fill="algo", color="algo"))
    + geom_density(alpha=0.75)
    + facet_grid(
        cols="descriptor",
        rows="setting",
        scales="free",
    )
    + scale_x_continuous(trans="log10")
    + coord_cartesian(ylim=(0, 10))
    + theme_minimal()
    + labs(
        x="Parameter value",
        y="Distribution density",
        color="Source",
        fill="Source",
        subtitle="(B) Individual parameter estimates for Mavoglurant model",
    )
    + scale_fill_cmap_d(color_scale)
    + scale_color_cmap_d(color_scale)
    + theme(
        figure_size=final_fig_size,
        plot_background=element_rect(fill="none"),
        legend_position="bottom",
    )
)
p1

## Performance benchmark data

In [ ]:
perf_nlmixr_PDU_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_nlmixr_PDU_runtime.csv")
perf_nlmixr_MI_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_nlmixr_MI_runtime.csv")
perf_Simwork_PDU_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_Simwork_PDU_runtime.csv")
#perf_Simwork_MI_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_Simwork_MI_runtime.csv")
perf_Simwork_surrogate_MI_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_Simwork_surrogate_MI_runtime.csv")
perf_Simwork_surrogate_PDU_raw = pd.read_csv("Benchmark_Data_Runtime/Mavoglurant_Simwork_surrogate_PDU_runtime.csv")

perf_nlmixr_PDU = perf_nlmixr_PDU_raw[["time_sec"]]
perf_nlmixr_PDU = perf_nlmixr_PDU.rename(columns= {"time_sec" : "total_time"})
perf_nlmixr_MI = perf_nlmixr_MI_raw[["time_sec"]]
perf_nlmixr_MI = perf_nlmixr_MI.rename(columns= {"time_sec" : "total_time"})
perf_nlmixr_PDU["algo"] = "nlmixr"
perf_nlmixr_MI["algo"] = "nlmixr"
perf_nlmixr_PDU["Param"] = "PDU"
perf_nlmixr_MI["Param"] = "MI"

perf_Simwork_PDU = perf_Simwork_PDU_raw.assign(algo="Simwork",Param="PDU").rename(columns={"time_saem":"total_time"})
#perf_Simwork_MI = perf_Simwork_MI_raw.assign(algo="Simwork",Param="MI")


perf_Simwork_surrogate_MI = perf_Simwork_surrogate_MI_raw[["total_time"]].assign(algo="Simwork_GP",Param="MI")
perf_Simwork_surrogate_PDU = perf_Simwork_surrogate_PDU_raw[["total_time"]].assign(algo="Simwork_GP",Param="PDU")

perf_df = pd.concat([
    perf_nlmixr_MI,
    perf_nlmixr_PDU,
    perf_Simwork_PDU,
    #perf_Simwork_MI,
    perf_Simwork_surrogate_MI,
    perf_Simwork_surrogate_PDU
])



In [ ]:
(
    ggplot(perf_df, aes(x="algo", y="total_time", fill="algo"))

    + geom_point(position=position_jitter(width=0.05), alpha=0.3)
    
    + geom_boxplot(width=0.6, alpha=0.8)

    + scale_y_log10()
    
    + facet_wrap("~Param", ncol=1)
    
    + theme_bw()
    
    + labs(x="", y="Runtime (s)")
)